# Dataset Inventory & Data Quality

Notebook này quét toàn bộ dataset hiện có trong `data/`, đối chiếu practice và redacted, rồi chỉ ra các điểm đáng chú ý cho team:

- số trip, số frame, độ dài, FPS
- mức độ đầy đủ của từng modality
- trip nào có/không có driver label hoặc risk ground truth
- coverage của image/depth/label/calibration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    if (PROJECT_ROOT.parent / "AGENTS.md").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("Cannot locate project root from the current notebook working directory.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.notebooks import fleetiq_notebook_utils as nb

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
inventory = nb.build_dataset_inventory()
inventory

In [ ]:
inventory.groupby("dataset_name")[
    [
        "frame_count",
        "duration_sec",
        "event_log_count",
        "left_images",
        "right_images",
        "driver_images",
        "depth_files",
        "label_files",
    ]
].agg(["count", "sum", "mean"])

In [ ]:
anomalies = inventory.assign(
    missing_driver_labels=lambda df: ~df["driver_labels_available"],
    missing_risk_gt=lambda df: ~df["risk_ground_truth_available"],
    partial_road_assets=lambda df: (df["left_image_coverage"] < 1.0) | (df["right_image_coverage"] < 1.0),
    sparse_depth=lambda df: df["depth_coverage"] < 1.0,
    sparse_labels=lambda df: df["label_coverage"] < 1.0,
)
anomalies[
    [
        "trip_id",
        "dataset_name",
        "frame_count",
        "driver_labels_available",
        "risk_ground_truth_available",
        "left_image_coverage",
        "right_image_coverage",
        "driver_image_coverage",
        "depth_coverage",
        "label_coverage",
        "partial_road_assets",
        "sparse_depth",
        "sparse_labels",
    ]
]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

inventory.plot.bar(x="trip_id", y="frame_count", color="#034EA2", ax=axes[0], legend=False)
axes[0].set_title("Frame count per trip")
axes[0].tick_params(axis="x", rotation=75)

coverage_cols = ["left_image_coverage", "driver_image_coverage", "depth_coverage", "label_coverage"]
inventory.set_index("trip_id")[coverage_cols].plot.bar(ax=axes[1])
axes[1].set_title("Modality coverage ratio")
axes[1].tick_params(axis="x", rotation=75)

inventory.groupby("dataset_name")[["driver_labels_available", "risk_ground_truth_available"]].mean().plot.bar(
    ax=axes[2], color=["#F37021", "#19226D"]
)
axes[2].set_title("Ground-truth availability")
axes[2].set_ylim(0, 1.05)
axes[2].legend(loc="lower right")

plt.tight_layout()

In [ ]:
trip_paths = nb.canonical_trip_paths()
trip_paths

In [ ]:
out_path = PROJECT_ROOT / "artifacts" / "dataset_inventory.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
inventory.to_csv(out_path, index=False)
out_path